# Análisis Exploratorio de Datos (EDA) — Proyecto NLP: Clasificación de Sentimiento

**Dataset:** `merged_hotels.json` — 680 hoteles | Barcelona · Madrid · Bilbao

Este notebook unifica tres análisis complementarios y los orienta hacia el objetivo final del proyecto: **entrenar un clasificador de sentimiento sobre las reseñas de hoteles**.

### Estructura del notebook

| Bloque | Contenido |
|--------|-----------|
| **A — Configuración** | Imports, rutas, stopwords, funciones comunes |
| **B — Reseñas** | Visión general, distribución de scores, análisis temporal, longitud de textos, n-gramas, word clouds, LDA, TF-IDF |
| **C — Servicios** | Distribución, frecuencia, macro-categorías, heatmap por ciudad, correlación con rating |
| **D — Precios** | Distribución, outliers, relación con rating y servicios |
| **E — Visión integrada** | Definición del target, balance del dataset, features estructurales → sentimiento, calidad del texto por clase |
| **F — Checklist pre-modelo** | Resumen de decisiones necesarias antes del entrenamiento |

---
## Bloque A — Configuración General

In [ ]:
import json, re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from wordcloud import WordCloud
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# ─── RUTAS ───────────────────────────────────────────────────────────────────
BASE_PATH  = "/Users/alessiadelconte/codici/progetto NPL"
INPUT_FILE = os.path.join(BASE_PATH, "DATASET/merged_hotels.json")
OUTPUT_DIR = os.path.join(BASE_PATH, "output_eda_completa_ES")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── PALETA DE COLORES (coherente en todo el notebook) ───────────────────────
CITY_COLORS = {'Barcelona': '#2196F3', 'Madrid': '#FF9800', 'Bilbao': '#4CAF50'}
SRC_COLORS  = {'booking': '#9C27B0', 'hotels_com': '#F44336', 'expedia': '#00BCD4'}
SENT_COLORS = {'Negativo': '#F44336', 'Neutro': '#FFC107', 'Positivo': '#4CAF50'}
CAT_COLOR   = '#5C6BC0'

# ─── STOPWORDS ───────────────────────────────────────────────────────────────
STOP_ES = set("""
a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante
e el ella ellas ello ellos en entre era eras éramos eran es esa esas ese eso esos esta estas
este esto estos fue fueron fui había hacía han has hasta he hemos hube hubo la las le les lo
los me mi mis muy más no nos nuestro nuestros o os otro otros para pero por porque que quien
se si sin sobre son su sus también te tenía todo todos tu tus un una unas unos ya yo
muy bien bueno buena buenos buenas es muy ha sido hotel habitación habitaciones todo todos
toda todas nada poco nada está están aquí allí tan hay ser estar tener
""".split())
STOP_EN = set("""
the a an and or but in on at to for of with is was are were be been having
have has had do does did will would could should may might shall can
this that these those it its i you he she we they me him her us them
my your his her our their all any both each few more most other some such no nor
only own same than then very just as also so if about after before between because
""".split())
STOPWORDS = STOP_ES | STOP_EN

# ─── MACRO-CATEGORÍAS DE SERVICIOS ───────────────────────────────────────────
CATEGORIES = {
    'WiFi':               ['wifi', 'wi-fi', 'internet', 'wireless'],
    'Desayuno':           ['desayun', 'breakfast', 'bufé', 'buffet'],
    'Parking':            ['aparcamiento', 'parking', 'garaje', 'garage'],
    'Restaurante/Bar':    ['restaurante', 'restaurant', 'bar', 'lounge', 'cafetería', 'café'],
    'Piscina':            ['piscina', 'pool', 'swimming'],
    'Spa/Fitness':        ['spa', 'wellness', 'sauna', 'gimnasio', 'fitness', 'gym'],
    'Mascotas':           ['mascota', 'pet', 'animales'],
    'Accesibilidad':      ['accesible', 'silla de ruedas', 'wheelchair', 'accessible'],
    'Aire Acond.':        ['aire acondicionado', 'air condition', 'climatiz'],
    'Lavandería':         ['lavandería', 'laundry', 'tintorería'],
    'Recepción 24h':      ['24 horas', '24h', '24-hour', '24 hours'],
    'Terraza/Jardín':     ['terraza', 'jardín', 'terrace', 'garden', 'balcón'],
    'No Fumadores':       ['fumadores', 'smoking', 'non-smoking'],
    'Traslado':           ['aeropuerto', 'airport', 'transfer', 'taxi'],
}

def classify_services(service_list):
    """Devuelve el conjunto de macro-categorías presentes en la lista de servicios."""
    cats = set()
    for svc in service_list:
        svc_low = svc.lower()
        for cat, keywords in CATEGORIES.items():
            if any(kw in svc_low for kw in keywords):
                cats.add(cat)
    return cats

def clean_text(text):
    """Limpieza básica para análisis de texto: minúsculas, sin números ni puntuación, sin stopwords."""
    text = str(text).lower()
    text = re.sub(r'\d+/\d+', '', text)
    text = re.sub(r'[^\w\sáéíóúüñà]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return tokens, ' '.join(tokens)

print(f"✅ Configuración lista. Directorio de salida: {OUTPUT_DIR}")

In [ ]:
# ─── CARGA DE DATOS Y CONSTRUCCIÓN DEL DATAFRAME INTEGRADO ───────────────────
# Una fila = una reseña, con todas las features del hotel adjunto

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"✅ Archivo cargado: {len(data)} hoteles")

rows_reviews = []   # nivel reseña (para Bloques B y E)
rows_hotels  = []   # nivel hotel (para Bloques C y D)
all_services_raw = []

for hotel in data:
    if not isinstance(hotel, dict): continue

    try:
        price = float(str(hotel.get('price', 0)).replace(',', '').strip())
        price = price if price > 0 else np.nan
    except:
        price = np.nan

    services = hotel.get('services', [])
    cats = classify_services(services)
    all_services_raw.extend(services)

    # ── DataFrame hoteles ──
    hotel_row = {
        'nombre_hotel': hotel.get('name', 'N/A'),
        'ciudad':       hotel.get('city', 'N/A'),
        'rating':       pd.to_numeric(hotel.get('rating'), errors='coerce'),
        'precio':       price,
        'n_servicios':  len(services),
        'n_reseñas':    len(hotel.get('reviews', [])),
        'n_categorias': len(cats),
    }
    for cat in CATEGORIES:
        col = f'has_{cat.replace("/","_").replace(" ","_").replace(".","")}'
        hotel_row[col] = int(cat in cats)
    rows_hotels.append(hotel_row)

    # ── DataFrame reseñas ──
    hotel_meta = {
        'nombre_hotel': hotel.get('name', 'N/A'),
        'ciudad':       hotel.get('city', 'N/A'),
        'rating_hotel': pd.to_numeric(hotel.get('rating'), errors='coerce'),
        'precio':       price,
        'n_servicios':  len(services),
        'has_Piscina':  int('Piscina' in cats),
        'has_Spa':      int('Spa/Fitness' in cats),
        'has_Desayuno': int('Desayuno' in cats),
        'has_Parking':  int('Parking' in cats),
        'has_Terraza':  int('Terraza/Jardín' in cats),
    }
    for rev in hotel.get('reviews', []):
        if isinstance(rev, dict):
            row = hotel_meta.copy()
            row.update({
                'comentario': str(rev.get('comment', '')),
                'score':      pd.to_numeric(rev.get('score'), errors='coerce'),
                'fuente':     rev.get('source', ''),
                'fecha':      rev.get('date', ''),
            })
            rows_reviews.append(row)

# DataFrames finales
df_h = pd.DataFrame(rows_hotels)   # 680 hoteles
df_r = pd.DataFrame(rows_reviews)  # reseñas

CAT_COLS_H = [c for c in df_h.columns if c.startswith('has_')]

# Limpieza reseñas
df_r = df_r.dropna(subset=['score'])
df_r = df_r[df_r['comentario'].str.len() > 5]
df_r['fecha']         = pd.to_datetime(df_r['fecha'], errors='coerce')
df_r['año']           = df_r['fecha'].dt.year
df_r['n_palabras']    = df_r['comentario'].str.split().str.len()
df_r['long_comment']  = df_r['comentario'].str.len()
df_r[['tokens', 'texto_limpio']] = df_r['comentario'].apply(
    lambda x: pd.Series(clean_text(x)))

# Variable target: sentimiento ternario
df_r['sentimiento'] = pd.cut(
    df_r['score'], bins=[0, 6, 8, 10],
    labels=['Negativo', 'Neutro', 'Positivo'])
df_r['sentimiento_bin'] = pd.cut(
    df_r['score'], bins=[0, 7, 10],
    labels=['Negativo', 'Positivo'])
df_r['franja_score'] = pd.cut(
    df_r['score'], bins=[0, 6, 8, 10],
    labels=['Bajo (≤6)', 'Medio (7-8)', 'Alto (9-10)'])

print(f"\n📊 DataFrame hoteles:  {df_h.shape[0]} filas × {df_h.shape[1]} columnas")
print(f"📊 DataFrame reseñas:  {df_r.shape[0]} filas × {df_r.shape[1]} columnas")
print(f"\nDistribución de scores:")
for score, cnt in df_r['score'].value_counts().sort_index().items():
    print(f"  Score {score:.0f}: {cnt:>4} reseñas ({cnt/len(df_r)*100:.1f}%)")

---
## Bloque B — Análisis de Reseñas

Visión general del corpus textual: volumen, distribución de scores, evolución temporal, longitud de los comentarios, frecuencia de términos y modelado de tópicos.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B1 — VISIÓN GENERAL: VOLUMEN DE RESEÑAS
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('B1 — Visión General del Corpus', fontsize=16, fontweight='bold')

# B1a: Reseñas por ciudad
city_rev = df_r.groupby('ciudad').size().sort_values(ascending=False)
bars = axes[0].bar(city_rev.index, city_rev.values,
                   color=[CITY_COLORS.get(c, '#607D8B') for c in city_rev.index], edgecolor='white')
for bar, val in zip(bars, city_rev.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 10,
                 f'{val}', ha='center', fontweight='bold')
axes[0].set_title('Reseñas por Ciudad', fontweight='bold')
axes[0].set_ylabel('N.º reseñas')

# B1b: Reseñas por plataforma
src_rev = df_r.groupby('fuente').size().sort_values(ascending=False)
bars = axes[1].bar(src_rev.index, src_rev.values,
                   color=[SRC_COLORS.get(s, '#607D8B') for s in src_rev.index], edgecolor='white')
for bar, val in zip(bars, src_rev.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 10,
                 f'{val}', ha='center', fontweight='bold')
axes[1].set_title('Reseñas por Plataforma', fontweight='bold')
axes[1].set_ylabel('N.º reseñas')

# B1c: Distribución reseñas por hotel
rev_per_hotel = df_r.groupby('nombre_hotel').size()
axes[2].hist(rev_per_hotel, bins=15, color='#607D8B', edgecolor='white')
axes[2].axvline(rev_per_hotel.mean(), color='red', linestyle='--',
                label=f'Media: {rev_per_hotel.mean():.1f}')
axes[2].set_title('Distribución Reseñas/Hotel', fontweight='bold')
axes[2].set_xlabel('N.º reseñas')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B1_vision_general.png'), dpi=150)
plt.close()
print("Fig B1 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B2 — DISTRIBUCIÓN DE SCORES
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('B2 — Distribución de Scores', fontsize=16, fontweight='bold')

# B2a: Distribución global
score_counts = df_r['score'].value_counts().sort_index()
bar_colors = ['#F44336','#F44336','#F44336','#FFC107','#4CAF50']
bars = axes[0].bar(score_counts.index.astype(str), score_counts.values,
                   color=bar_colors, edgecolor='white')
axes[0].axvline(str(df_r['score'].mean()), color='red', linestyle='--',
                label=f'Media: {df_r["score"].mean():.2f}')
for bar, val in zip(bars, score_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 20,
                 f'{val}', ha='center', fontsize=8)
axes[0].set_title('Distribución Global de Scores', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].legend()

# B2b: Boxplot por ciudad
bp_data = [df_r[df_r['ciudad']==c]['score'].dropna().values
           for c in ['Barcelona','Madrid','Bilbao']]
bp = axes[1].boxplot(bp_data, labels=['Barcelona','Madrid','Bilbao'], patch_artist=True)
for patch, city in zip(bp['boxes'], ['Barcelona','Madrid','Bilbao']):
    patch.set_facecolor(CITY_COLORS[city])
    patch.set_alpha(0.7)
for i, city in enumerate(['Barcelona','Madrid','Bilbao']):
    mean_v = df_r[df_r['ciudad']==city]['score'].mean()
    axes[1].text(i+1, mean_v - 0.4, f'μ={mean_v:.2f}', ha='center', fontsize=8, color='red')
axes[1].set_title('Score por Ciudad', fontweight='bold')
axes[1].set_ylabel('Score')

# B2c: Score medio por plataforma
score_src = df_r.groupby('fuente')['score'].mean().sort_values()
bars = axes[2].barh(score_src.index, score_src.values,
                    color=[SRC_COLORS.get(s,'#607D8B') for s in score_src.index], edgecolor='white')
axes[2].set_xlim(score_src.min() - 0.5, 10.3)
for bar, val in zip(bars, score_src.values):
    axes[2].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=9)
axes[2].set_title('Score Medio por Plataforma\n⚠️ Booking: solo scores positivos', fontweight='bold')
axes[2].set_xlabel('Score medio')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B2_scores.png'), dpi=150)
plt.close()
print("Fig B2 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B3 — ANÁLISIS TEMPORAL
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('B3 — Evolución Temporal de las Reseñas', fontsize=16, fontweight='bold')

df_time = df_r.dropna(subset=['año'])
df_time = df_time[df_time['año'] >= 2019]

# B3a: Volumen por año y ciudad
for city, color in CITY_COLORS.items():
    yearly = df_time[df_time['ciudad']==city].groupby('año').size()
    axes[0].plot(yearly.index, yearly.values, marker='o', label=city, color=color, linewidth=2)
axes[0].set_title('Volumen de Reseñas por Año', fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('N.º reseñas')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# B3b: Score medio en el tiempo
score_year = df_time.groupby('año')['score'].mean()
axes[1].plot(score_year.index, score_year.values, marker='s',
             color='#E91E63', linewidth=2.5)
axes[1].fill_between(score_year.index, score_year.values,
                     alpha=0.15, color='#E91E63')
axes[1].set_title('Evolución del Score Medio', fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Score medio')
axes[1].set_ylim(8, 10.5)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B3_temporal.png'), dpi=150)
plt.close()
print("Fig B3 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B4 — ANÁLISIS DE LONGITUD DE TEXTOS (PRE-NLP)
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('B4 — Longitud de los Comentarios', fontsize=16, fontweight='bold')

# B4a: Histograma global de palabras
axes[0].hist(df_r['n_palabras'].clip(upper=100), bins=30, color='#3F51B5', edgecolor='white')
axes[0].axvline(df_r['n_palabras'].mean(), color='red', linestyle='--',
                label=f'Media: {df_r["n_palabras"].mean():.1f} palabras')
axes[0].axvline(df_r['n_palabras'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df_r["n_palabras"].median():.0f} palabras')
axes[0].set_title('Distribución N.º de Palabras (cap. 100)', fontweight='bold')
axes[0].set_xlabel('N.º de palabras')
axes[0].legend()

# B4b: Palabras medias por franja de score
wc_bins = df_r.groupby('franja_score', observed=True)['n_palabras'].mean()
bars = axes[1].bar(wc_bins.index, wc_bins.values,
                   color=['#F44336','#FFC107','#4CAF50'], edgecolor='white')
for bar, val in zip(bars, wc_bins.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val:.1f}', ha='center', fontweight='bold')
axes[1].set_title('Palabras Medias por Franja de Score\n(los negativos escriben más)', fontweight='bold')
axes[1].set_ylabel('N.º palabras medio')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B4_longitud_textos.png'), dpi=150)
plt.close()
print("Fig B4 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B5 — N-GRAMAS: BIGRAMAS MÁS FRECUENTES
# ══════════════════════════════════════════════════════════════════════════
def get_top_ngrams(corpus, n=2, k=15):
    vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
    bow = vec.transform(corpus)
    sum_words = bow.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:k]

top_bigrams = get_top_ngrams(df_r['texto_limpio'], n=2)
b_words, b_counts = zip(*top_bigrams)

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('B5 — Top 15 Bigramas (pares de palabras más frecuentes)', fontsize=14, fontweight='bold')
ax.barh(b_words[::-1], b_counts[::-1], color='#388E3C', edgecolor='white')
ax.set_xlabel('Frecuencia')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B5_ngramas.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig B5 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B6 — WORD CLOUDS POR CIUDAD
# ══════════════════════════════════════════════════════════════════════════
CITY_WC_COLORS = {'Barcelona': 'Blues', 'Madrid': 'Oranges', 'Bilbao': 'Greens'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('B6 — Nube de Palabras por Ciudad', fontsize=16, fontweight='bold')

for ax, city in zip(axes, ['Barcelona','Madrid','Bilbao']):
    text = ' '.join(df_r[df_r['ciudad']==city]['texto_limpio'])
    if len(text.strip()) < 10:
        ax.text(0.5, 0.5, 'Sin datos suficientes', ha='center', va='center')
    else:
        wc = WordCloud(width=500, height=400, background_color='white',
                       max_words=60, colormap=CITY_WC_COLORS[city]).generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.set_title(city, fontsize=14, fontweight='bold', color=CITY_COLORS[city])
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B6_wordclouds_ciudad.png'), dpi=150)
plt.close()
print("Fig B6 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B7 — MODELADO DE TÓPICOS (LDA)
# ══════════════════════════════════════════════════════════════════════════
vectorizer_lda = CountVectorizer(max_df=0.9, min_df=5, max_features=500)
X_lda = vectorizer_lda.fit_transform(df_r['texto_limpio'])
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X_lda)

words_lda = vectorizer_lda.get_feature_names_out()
TOPIC_NAMES = ['Tópico 1', 'Tópico 2', 'Tópico 3', 'Tópico 4', 'Tópico 5']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('B7 — Modelado de Tópicos LDA (5 temas latentes)', fontsize=16, fontweight='bold')

for i, (topic, ax) in enumerate(zip(lda.components_, axes)):
    top_idx = topic.argsort()[-10:]
    ax.barh([words_lda[j] for j in top_idx], [topic[j] for j in top_idx], color='teal')
    ax.set_title(TOPIC_NAMES[i], fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B7_lda_topicos.png'), dpi=150)
plt.close()
print("Fig B7 guardada ✓")

# Asignar tópico dominante
topic_values = lda.transform(X_lda)
df_r['topico_dominante'] = topic_values.argmax(axis=1)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B8 — TF-IDF: PALABRAS CORRELADAS CON SCORE
# ══════════════════════════════════════════════════════════════════════════
tfidf = TfidfVectorizer(max_features=100)
X_tfidf = tfidf.fit_transform(df_r['texto_limpio'])
corrs_tfidf = [
    np.corrcoef(X_tfidf[:, i].toarray().flatten(), df_r['score'].fillna(0))[0, 1]
    for i in range(X_tfidf.shape[1])
]

feat_names = tfidf.get_feature_names_out()
sorted_idx = np.argsort(corrs_tfidf)

fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle('B8 — Palabras más correladas con Score Alto (verde) y Bajo (rojo)',
             fontsize=14, fontweight='bold')
ax.barh([feat_names[i] for i in sorted_idx[-15:]], [corrs_tfidf[i] for i in sorted_idx[-15:]],
        color='#4CAF50', edgecolor='white')
ax.barh([feat_names[i] for i in sorted_idx[:15]], [corrs_tfidf[i] for i in sorted_idx[:15]],
        color='#F44336', edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente de correlación con Score')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B8_tfidf_correlacion.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig B8 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG B9 — DISTRIBUCIÓN DE TÓPICOS POR CIUDAD
# ══════════════════════════════════════════════════════════════════════════
city_topics = pd.crosstab(df_r['ciudad'], df_r['topico_dominante'], normalize='index')
city_topics.columns = TOPIC_NAMES

city_topics.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis')
plt.title('B9 — Distribución de Tópicos LDA por Ciudad', fontsize=14, fontweight='bold')
plt.ylabel('Proporción')
plt.xlabel('')
plt.xticks(rotation=0)
plt.legend(title='Tópico', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'B9_topicos_por_ciudad.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig B9 guardada ✓")

---
## Bloque C — Análisis de Servicios

Distribución del número de servicios por hotel, frecuencia de servicios individuales, prevalencia de macro-categorías y diferencias entre ciudades.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG C1 — VISIÓN GENERAL DE SERVICIOS
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('C1 — Visión General de Servicios', fontsize=16, fontweight='bold')

# C1a: Histograma n_servicios
axes[0].hist(df_h['n_servicios'], bins=25, color=CAT_COLOR, edgecolor='white')
axes[0].axvline(df_h['n_servicios'].mean(), color='red', linestyle='--',
                label=f'Media: {df_h["n_servicios"].mean():.1f}')
axes[0].axvline(df_h['n_servicios'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df_h["n_servicios"].median():.1f}')
axes[0].set_title('Distribución N.º Servicios por Hotel', fontweight='bold')
axes[0].set_xlabel('N.º servicios')
axes[0].legend()

# C1b: Boxplot por ciudad
bp_data = [df_h[df_h['ciudad']==c]['n_servicios'].values for c in ['Barcelona','Madrid','Bilbao']]
bp = axes[1].boxplot(bp_data, labels=['Barcelona','Madrid','Bilbao'], patch_artist=True)
for patch, city in zip(bp['boxes'], ['Barcelona','Madrid','Bilbao']):
    patch.set_facecolor(CITY_COLORS[city])
    patch.set_alpha(0.7)
axes[1].set_title('Servicios por Ciudad (boxplot)', fontweight='bold')
axes[1].set_ylabel('N.º servicios')

# C1c: Scatter n_servicios vs n_categorias
for city, color in CITY_COLORS.items():
    sub = df_h[df_h['ciudad'] == city]
    axes[2].scatter(sub['n_servicios'], sub['n_categorias'],
                    color=color, alpha=0.5, s=20, label=city)
axes[2].set_title('N.º Servicios vs N.º Categorías', fontweight='bold')
axes[2].set_xlabel('N.º servicios (cadenas brutas)')
axes[2].set_ylabel('N.º macro-categorías cubiertas')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'C1_overview_servicios.png'), dpi=150)
plt.close()
print("Fig C1 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG C2 — TOP 25 SERVICIOS POR FRECUENCIA
# ══════════════════════════════════════════════════════════════════════════
cnt_svc = Counter(all_services_raw)
top_n = 25
top_svcs, top_counts = zip(*cnt_svc.most_common(top_n))

fig, ax = plt.subplots(figsize=(12, 9))
fig.suptitle('C2 — Top 25 Servicios por Frecuencia', fontsize=16, fontweight='bold')

bars = ax.barh(range(top_n), top_counts[::-1], color=CAT_COLOR, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([s[:65] for s in top_svcs[::-1]], fontsize=8)
ax.set_xlabel('N.º hoteles que ofrecen el servicio')
ax.set_title('Nota: algunas entradas similares son versiones multiidioma (es/en) del mismo servicio',
             fontsize=9, color='grey')

for bar, val in zip(bars, top_counts[::-1]):
    ax.text(val + 3, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/len(df_h)*100:.0f}%)', va='center', fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'C2_top_servicios.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig C2 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG C3 — PREVALENCIA DE MACRO-CATEGORÍAS
# ══════════════════════════════════════════════════════════════════════════
cat_labels = [c.replace('has_','').replace('_',' ') for c in CAT_COLS_H]
cat_pct    = [df_h[c].mean() * 100 for c in CAT_COLS_H]
sorted_pairs = sorted(zip(cat_pct, cat_labels), reverse=True)
cat_pct_s, cat_labels_s = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('C3 — Prevalencia de Macro-Categorías de Servicios', fontsize=16, fontweight='bold')

colors_c3 = ['#4CAF50' if p >= 50 else '#FFC107' if p >= 25 else '#F44336' for p in cat_pct_s]
bars = ax.barh(cat_labels_s, cat_pct_s, color=colors_c3, edgecolor='white')
ax.set_xlabel('% hoteles que ofrecen la categoría')
ax.axvline(50, color='grey', linestyle='--', linewidth=0.8)

for bar, val in zip(bars, cat_pct_s):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

patches = [
    mpatches.Patch(color='#4CAF50', label='≥ 50% hoteles'),
    mpatches.Patch(color='#FFC107', label='25–50%'),
    mpatches.Patch(color='#F44336', label='< 25%'),
]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'C3_prevalencia_categorias.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig C3 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG C4 — HEATMAP CATEGORÍAS POR CIUDAD
# ══════════════════════════════════════════════════════════════════════════
city_cat_matrix = df_h.groupby('ciudad')[CAT_COLS_H].mean() * 100
city_cat_matrix.columns = cat_labels

fig, ax = plt.subplots(figsize=(14, 4))
fig.suptitle('C4 — Difusión de Categorías por Ciudad (%)', fontsize=16, fontweight='bold')

im = ax.imshow(city_cat_matrix.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)
ax.set_xticks(range(len(cat_labels)))
ax.set_xticklabels(cat_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(3))
ax.set_yticklabels(city_cat_matrix.index)

for i in range(len(city_cat_matrix.index)):
    for j in range(len(cat_labels)):
        val = city_cat_matrix.values[i, j]
        color = 'white' if val > 65 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='% hoteles', shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'C4_heatmap_categorias_ciudad.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig C4 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG C5 — SERVICIOS vs RATING
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('C5 — Servicios vs Rating del Hotel', fontsize=16, fontweight='bold')

# C5a: Scatter n_servicios vs rating
for city, color in CITY_COLORS.items():
    sub = df_h[df_h['ciudad'] == city].dropna(subset=['rating'])
    axes[0].scatter(sub['n_servicios'], sub['rating'], color=color, alpha=0.45, s=18, label=city)
corr_sr = df_h['n_servicios'].corr(df_h['rating'])
axes[0].set_title(f'N.º Servicios vs Rating (r={corr_sr:.3f})', fontweight='bold')
axes[0].set_xlabel('N.º servicios')
axes[0].set_ylabel('Rating')
axes[0].legend(fontsize=8)

# C5b: Rating medio por franja de servicios
df_h['franja_servicios'] = pd.cut(df_h['n_servicios'], bins=[0,10,30,60,130],
    labels=['Bajo (0-10)','Medio (11-30)','Alto (31-60)','Muy Alto (61+)'])
fascia_rating = df_h.groupby('franja_servicios', observed=True)['rating'].mean()
bars = axes[1].bar(fascia_rating.index, fascia_rating.values,
                   color=['#EF9A9A','#FFCC80','#A5D6A7','#90CAF9'], edgecolor='white')
axes[1].set_ylim(7, 9.5)
for bar, val in zip(bars, fascia_rating.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.2f}', ha='center', fontweight='bold')
axes[1].set_title('Rating Medio por Franja de Servicios', fontweight='bold')
axes[1].set_ylabel('Rating medio')

# C5c: Correlación categoría → rating
cat_rating_corr = [(col.replace('has_','').replace('_',' '), df_h[col].corr(df_h['rating']))
                   for col in CAT_COLS_H]
cat_rating_corr.sort(key=lambda x: x[1])
labels_cr, corrs_cr = zip(*cat_rating_corr)
colors_cr = ['#4CAF50' if c >= 0 else '#F44336' for c in corrs_cr]
axes[2].barh(labels_cr, corrs_cr, color=colors_cr, edgecolor='white')
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('Correlación Categoría → Rating', fontweight='bold')
axes[2].set_xlabel('Coeficiente de correlación')
axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'C5_servicios_vs_rating.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig C5 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG C6 — WORD CLOUD DE SERVICIOS POR CIUDAD
# ══════════════════════════════════════════════════════════════════════════
STOP_SVC = set("""
se de las los un una el la en con para por que hay puede disponible algunas
zonas comunes gratis todas habitaciones hotel alojamiento instalaciones
servicio servicios tienen tienes tiene si no también
""".split())

def clean_svc_text(service_list):
    text = ' '.join(service_list).lower()
    text = re.sub(r'[^\w\sáéíóúüñ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return ' '.join(w for w in text.split() if w not in STOP_SVC and len(w) > 3)

CITY_WC = {'Barcelona': 'Blues', 'Madrid': 'Oranges', 'Bilbao': 'Greens'}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('C6 — Nube de Palabras de Servicios por Ciudad', fontsize=16, fontweight='bold')

for ax, city in zip(axes, ['Barcelona','Madrid','Bilbao']):
    city_services = []
    for hotel in data:
        if hotel.get('city') == city:
            city_services.extend(hotel.get('services', []))
    clean = clean_svc_text(city_services)
    wc = WordCloud(width=500, height=400, background_color='white',
                   max_words=60, colormap=CITY_WC[city]).generate(clean)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(city, fontsize=14, fontweight='bold', color=CITY_COLORS[city])
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'C6_wordcloud_servicios.png'), dpi=150)
plt.close()
print("Fig C6 guardada ✓")

---
## Bloque D — Análisis de Precios

**Nota metodológica:** el campo `price` representa el precio medio por noche (en EUR) extraído en el momento del scraping. No es una media anual, por lo que debe tratarse como indicativo. Se detectan outliers significativos, especialmente en Bilbao.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG D1 — DISTRIBUCIÓN DE PRECIOS
# ══════════════════════════════════════════════════════════════════════════
PRICE_CAP = 600
df_p  = df_h.dropna(subset=['precio'])
df_pc = df_p[df_p['precio'] <= PRICE_CAP]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('D1 — Distribución de Precios por Noche (€)', fontsize=16, fontweight='bold')

# D1a: Histograma global (sin outliers)
axes[0].hist(df_pc['precio'], bins=30, color='#5C6BC0', edgecolor='white')
axes[0].axvline(df_pc['precio'].mean(), color='red', linestyle='--',
                label=f'Media: {df_pc["precio"].mean():.0f}€')
axes[0].axvline(df_pc['precio'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df_pc["precio"].median():.0f}€')
axes[0].set_title(f'Distribución Global (sin outliers >600€)', fontweight='bold')
axes[0].set_xlabel('Precio por noche (€)')
axes[0].legend()

# D1b: Boxplot por ciudad (todos los datos)
bp_data = [df_p[df_p['ciudad']==c]['precio'].values for c in ['Barcelona','Madrid','Bilbao']]
bp = axes[1].boxplot(bp_data, labels=['Barcelona','Madrid','Bilbao'], patch_artist=True, showfliers=True)
for patch, city in zip(bp['boxes'], ['Barcelona','Madrid','Bilbao']):
    patch.set_facecolor(CITY_COLORS[city])
    patch.set_alpha(0.7)
axes[1].set_title('Precio por Ciudad (todos los datos)', fontweight='bold')
axes[1].set_ylabel('€ / noche')

# D1c: Precio medio por ciudad
city_price = df_pc.groupby('ciudad')['precio'].mean().sort_values(ascending=False)
bars = axes[2].bar(city_price.index, city_price.values,
                   color=[CITY_COLORS[c] for c in city_price.index], edgecolor='white')
for bar, val in zip(bars, city_price.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 2,
                 f'{val:.0f}€', ha='center', fontweight='bold')
axes[2].set_title('Precio Medio por Ciudad', fontweight='bold')
axes[2].set_ylabel('€ / noche')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'D1_distribucion_precios.png'), dpi=150)
plt.close()
print("Fig D1 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG D2 — PRECIO vs RATING
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('D2 — Precio vs Rating', fontsize=16, fontweight='bold')

df_pr = df_pc.dropna(subset=['rating'])

# D2a: Scatter por ciudad con trendline
for city, color in CITY_COLORS.items():
    sub = df_pr[df_pr['ciudad'] == city]
    axes[0].scatter(sub['precio'], sub['rating'], color=color, alpha=0.5, s=20, label=city)
corr_pr = df_pr['precio'].corr(df_pr['rating'])
z = np.polyfit(df_pr['precio'], df_pr['rating'], 1)
x_line = np.linspace(df_pr['precio'].min(), df_pr['precio'].max(), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=1.5, label=f'Tendencia (r={corr_pr:.3f})')
axes[0].set_title('Precio vs Rating por Hotel', fontweight='bold')
axes[0].set_xlabel('Precio por noche (€)')
axes[0].set_ylabel('Rating')
axes[0].legend(fontsize=8)

# D2b: Rating medio por franja de precio
df_pr['franja_precio'] = pd.cut(df_pr['precio'], bins=[0,100,150,200,300,700],
    labels=['<100€','100-150€','150-200€','200-300€','>300€'])
fascia_r = df_pr.groupby('franja_precio', observed=True)['rating'].mean()
bars = axes[1].bar(fascia_r.index, fascia_r.values,
                   color=['#EF9A9A','#FFCC80','#FFF176','#A5D6A7','#90CAF9'], edgecolor='white')
axes[1].set_ylim(7.5, 9.5)
for bar, val in zip(bars, fascia_r.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.2f}', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Rating Medio por Franja de Precio\n(tendencia creciente monotónica)', fontweight='bold')
axes[1].set_ylabel('Rating medio')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'D2_precio_vs_rating.png'), dpi=150)
plt.close()
print("Fig D2 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG D3 — OUTLIERS Y DISTRIBUCIÓN EN VIOLÍN
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('D3 — Outliers de Precios y Distribución en Violín', fontsize=16, fontweight='bold')

# D3a: Detección de outliers (IQR)
Q1, Q3 = df_p['precio'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
df_p = df_p.copy()
df_p['es_outlier'] = df_p['precio'] > upper_fence

for city, color in CITY_COLORS.items():
    sub = df_p[df_p['ciudad'] == city]
    normal  = sub[~sub['es_outlier']]
    outlier = sub[sub['es_outlier']]
    axes[0].scatter(normal.index,  normal['precio'],  color=color, alpha=0.4, s=12, label=city)
    axes[0].scatter(outlier.index, outlier['precio'], color=color, alpha=0.9,
                    s=60, edgecolors='black', linewidths=1, marker='D')
axes[0].axhline(upper_fence, color='red', linestyle='--', label=f'Límite IQR: {upper_fence:.0f}€')
axes[0].set_title(f'Outliers (diamantes) — umbral IQR: {upper_fence:.0f}€', fontweight='bold')
axes[0].set_xlabel('Índice hotel')
axes[0].set_ylabel('Precio (€)')
axes[0].legend(fontsize=8)

# D3b: Violín por ciudad
data_violin = [df_p[df_p['ciudad']==c]['precio'].clip(upper=PRICE_CAP).values
               for c in ['Barcelona','Madrid','Bilbao']]
parts = axes[1].violinplot(data_violin, positions=[1,2,3], showmedians=True)
for pc, city in zip(parts['bodies'], ['Barcelona','Madrid','Bilbao']):
    pc.set_facecolor(CITY_COLORS[city])
    pc.set_alpha(0.6)
axes[1].set_xticks([1,2,3])
axes[1].set_xticklabels(['Barcelona','Madrid','Bilbao'])
axes[1].set_title(f'Distribución Precios por Ciudad\n(cap a {PRICE_CAP}€)', fontweight='bold')
axes[1].set_ylabel('€ / noche')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'D3_outliers_violin.png'), dpi=150)
plt.close()
print("Fig D3 guardada ✓")
print(f"\n📊 Outliers detectados (> {upper_fence:.0f}€): {df_p['es_outlier'].sum()} hoteles")
print(df_p[df_p['es_outlier']][['nombre_hotel','ciudad','precio','rating']].to_string())

---
## Bloque E — Visión Integrada: Features Estructurales → Sentimiento

Este bloque responde a la pregunta central del proyecto:
> *¿Las características estructurales del hotel (precio, servicios, ciudad, plataforma) son predictivas del sentimiento en las reseñas?*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG E1 — DEFINICIÓN DE LA VARIABLE OBJETIVO
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('E1 — Definición de la Variable Objetivo (Target)', fontsize=16, fontweight='bold')

# E1a: Scores brutos coloreados por clase
score_counts = df_r['score'].value_counts().sort_index()
bar_colors_e = ['#F44336','#F44336','#F44336','#FFC107','#4CAF50']
bars = axes[0].bar(score_counts.index.astype(str), score_counts.values,
                   color=bar_colors_e, edgecolor='white')
for bar, val in zip(bars, score_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 15,
                 f'{val}\n({val/len(df_r)*100:.1f}%)', ha='center', fontsize=8)
axes[0].set_title('Scores Brutos (discrete: 2,4,6,8,10)', fontweight='bold')
axes[0].set_xlabel('Score')
patches_leg = [
    mpatches.Patch(color='#F44336', label='Negativo (≤6)'),
    mpatches.Patch(color='#FFC107', label='Neutro (7-8)'),
    mpatches.Patch(color='#4CAF50', label='Positivo (9-10)'),
]
axes[0].legend(handles=patches_leg, fontsize=8)

# E1b: Pie ternario
sent_counts = df_r['sentimiento'].value_counts()
axes[1].pie(
    [sent_counts.get('Negativo',0), sent_counts.get('Neutro',0), sent_counts.get('Positivo',0)],
    labels=['Negativo\n(≤6)', 'Neutro\n(7-8)', 'Positivo\n(9-10)'],
    colors=['#F44336','#FFC107','#4CAF50'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Target Ternario\n(utilizado en el análisis)', fontweight='bold')

# E1c: Pie binario
bin_counts = df_r['sentimiento_bin'].value_counts()
axes[2].pie(
    [bin_counts.get('Negativo',0), bin_counts.get('Positivo',0)],
    labels=['Negativo\n(≤7)', 'Positivo\n(>7)'],
    colors=['#F44336','#4CAF50'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('Target Binario\n(alternativa para el modelo)', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'E1_definicion_target.png'), dpi=150)
plt.close()
print("Fig E1 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG E2 — BALANCE DEL DATASET POR CIUDAD Y PLATAFORMA
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('E2 — Balance del Dataset por Ciudad y Plataforma', fontsize=16, fontweight='bold')

# E2a: % sentimiento por ciudad
city_sent = pd.crosstab(df_r['ciudad'], df_r['sentimiento'], normalize='index') * 100
city_sent = city_sent.reindex(columns=['Negativo','Neutro','Positivo'], fill_value=0)
city_sent.plot(kind='bar', stacked=True, ax=axes[0],
               color=['#F44336','#FFC107','#4CAF50'], edgecolor='white')
axes[0].set_title('Sentimiento por Ciudad', fontweight='bold')
axes[0].set_ylabel('% reseñas')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Sentimiento', fontsize=8)

# E2b: % sentimiento por plataforma
src_sent = pd.crosstab(df_r['fuente'], df_r['sentimiento'], normalize='index') * 100
src_sent = src_sent.reindex(columns=['Negativo','Neutro','Positivo'], fill_value=0)
src_sent.plot(kind='bar', stacked=True, ax=axes[1],
              color=['#F44336','#FFC107','#4CAF50'], edgecolor='white')
axes[1].set_title('Sentimiento por Plataforma\n⚠️ Booking: 0% negativos', fontweight='bold',
                  color='#B71C1C')
axes[1].set_ylabel('% reseñas')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sentimiento', fontsize=8)

# E2c: N absoluto por clase y ciudad
city_sent_abs = pd.crosstab(df_r['ciudad'], df_r['sentimiento'])
city_sent_abs = city_sent_abs.reindex(columns=['Negativo','Neutro','Positivo'], fill_value=0)
city_sent_abs.plot(kind='bar', ax=axes[2],
                   color=['#F44336','#FFC107','#4CAF50'], edgecolor='white')
axes[2].axhline(100, color='red', linestyle='--', linewidth=1, label='Umbral mínimo 100')
axes[2].set_title('N.º Ejemplos Absolutos por Clase y Ciudad', fontweight='bold')
axes[2].set_ylabel('N.º reseñas')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'E2_balance_dataset.png'), dpi=150)
plt.close()
print("Fig E2 guardada ✓")
print("\n📊 N.º ejemplos por clase y ciudad:")
print(city_sent_abs.to_string())
print("\n📊 N.º ejemplos por clase y plataforma:")
print(pd.crosstab(df_r['fuente'], df_r['sentimiento']).to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG E3 — PRECIO → SENTIMIENTO
# ══════════════════════════════════════════════════════════════════════════
df_rp = df_r[df_r['precio'].notna() & (df_r['precio'] <= PRICE_CAP)].copy()
df_rp['franja_precio'] = pd.cut(df_rp['precio'], bins=[0,100,150,200,300,700],
    labels=['<100€','100-150€','150-200€','200-300€','>300€'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('E3 — Precio → Sentimiento', fontsize=16, fontweight='bold')

# E3a: % sentimiento por franja de precio
fp_sent = pd.crosstab(df_rp['franja_precio'], df_rp['sentimiento'], normalize='index') * 100
fp_sent = fp_sent.reindex(columns=['Negativo','Neutro','Positivo'], fill_value=0)
fp_sent.plot(kind='bar', stacked=True, ax=axes[0],
             color=['#F44336','#FFC107','#4CAF50'], edgecolor='white')
axes[0].set_title('Sentimiento por Franja de Precio', fontweight='bold')
axes[0].set_ylabel('% reseñas')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(fontsize=8)

# E3b: Score medio por franja de precio
score_fp = df_rp.groupby('franja_precio', observed=True)['score'].mean()
bars = axes[1].bar(score_fp.index, score_fp.values,
                   color=['#EF9A9A','#FFCC80','#FFF176','#A5D6A7','#90CAF9'], edgecolor='white')
axes[1].set_ylim(8.5, 10.2)
for bar, val in zip(bars, score_fp.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.2f}', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Score Medio por Franja de Precio\n(tendencia creciente monotónica)', fontweight='bold')
axes[1].set_ylabel('Score medio')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'E3_precio_vs_sentimiento.png'), dpi=150)
plt.close()
print("Fig E3 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG E4 — SERVICIOS → SENTIMIENTO
# ══════════════════════════════════════════════════════════════════════════
svc_feat_cols = [c for c in df_r.columns if c.startswith('has_')]

cat_effect = []
for col in svc_feat_cols:
    label = col.replace('has_','').replace('_',' ')
    pct_pos_yes = (df_r[df_r[col]==1]['sentimiento'] == 'Positivo').mean() * 100
    pct_pos_no  = (df_r[df_r[col]==0]['sentimiento'] == 'Positivo').mean() * 100
    cat_effect.append({'servicio': label, 'con': pct_pos_yes, 'sin': pct_pos_no,
                       'diferencia': pct_pos_yes - pct_pos_no})

df_eff = pd.DataFrame(cat_effect).sort_values('diferencia', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('E4 — Servicios → Sentimiento Positivo (%)', fontsize=16, fontweight='bold')

# E4a: Diferencia (con - sin)
colors_eff = ['#4CAF50' if d >= 0 else '#F44336' for d in df_eff['diferencia']]
bars = axes[0].barh(df_eff['servicio'], df_eff['diferencia'], color=colors_eff, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Δ % Positivo: Con vs Sin Servicio\n(verde = el servicio mejora el sentimiento)',
                  fontweight='bold')
axes[0].set_xlabel('Diferencia en puntos porcentuales')
for bar, val in zip(bars, df_eff['diferencia']):
    axes[0].text(val + (0.3 if val >= 0 else -0.3),
                 bar.get_y() + bar.get_height()/2,
                 f'{val:+.1f}pp', va='center', fontsize=9,
                 ha='left' if val >= 0 else 'right')

# E4b: % positivo con vs sin (barras agrupadas)
x = np.arange(len(df_eff))
w = 0.35
axes[1].barh(x - w/2, df_eff['con'], w, label='Con el servicio', color='#4CAF50', alpha=0.8)
axes[1].barh(x + w/2, df_eff['sin'], w, label='Sin el servicio', color='#90A4AE', alpha=0.8)
axes[1].set_yticks(x)
axes[1].set_yticklabels(df_eff['servicio'])
axes[1].set_title('% Reseñas Positivas: Con vs Sin Servicio', fontweight='bold')
axes[1].set_xlabel('% reseñas positivas')
axes[1].legend(fontsize=9)
axes[1].set_xlim(60, 105)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'E4_servicios_vs_sentimiento.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig E4 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG E5 — CALIDAD DEL TEXTO POR CLASE
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('E5 — Características del Texto por Clase de Sentimiento', fontsize=16, fontweight='bold')

# E5a: Histograma densidad por clase
for sent, color in SENT_COLORS.items():
    sub = df_r[df_r['sentimiento'] == sent]['n_palabras'].clip(upper=100)
    if len(sub) > 0:
        axes[0].hist(sub, bins=30, alpha=0.5, color=color,
                     label=f'{sent} (μ={sub.mean():.1f})', density=True)
axes[0].set_title('Distribución Longitud Comentarios por Clase', fontweight='bold')
axes[0].set_xlabel('N.º palabras')
axes[0].set_ylabel('Densidad')
axes[0].legend(fontsize=9)

# E5b: Boxplot palabras por clase
bp_data_s = [df_r[df_r['sentimiento']==s]['n_palabras'].clip(upper=120).values
             for s in ['Negativo','Neutro','Positivo']]
bp = axes[1].boxplot(bp_data_s, labels=['Negativo','Neutro','Positivo'],
                     patch_artist=True, showmeans=True)
for patch, color in zip(bp['boxes'], ['#F44336','#FFC107','#4CAF50']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for i, s in enumerate(['Negativo','Neutro','Positivo']):
    mean_v = df_r[df_r['sentimiento']==s]['n_palabras'].mean()
    axes[1].text(i+1, mean_v + 1, f'μ={mean_v:.1f}', ha='center', fontsize=8, color='red')
axes[1].set_title('N.º Palabras por Clase (boxplot)', fontweight='bold')
axes[1].set_ylabel('N.º palabras')

# E5c: % comentarios cortos (< 10 palabras) por clase
df_r['es_corto'] = df_r['n_palabras'] < 10
short_by_sent = df_r.groupby('sentimiento', observed=True)['es_corto'].mean() * 100
short_by_sent = short_by_sent.reindex(['Negativo','Neutro','Positivo'])
bars = axes[2].bar(short_by_sent.index, short_by_sent.values,
                   color=['#F44336','#FFC107','#4CAF50'], edgecolor='white')
for bar, val in zip(bars, short_by_sent.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[2].set_title('% Comentarios Cortos (<10 palabras) por Clase\n(proxy de reseñas poco informativas)',
                  fontweight='bold')
axes[2].set_ylabel('% reseñas cortas')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'E5_texto_por_clase.png'), dpi=150)
plt.close()
print("Fig E5 guardada ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIG E6 — VISIÓN UNIFICADA: CORRELACIÓN FEATURES ESTRUCTURALES → SCORE
# ══════════════════════════════════════════════════════════════════════════
feature_cols = ['precio', 'n_servicios', 'n_palabras'] + svc_feat_cols
df_corr = df_r[feature_cols + ['score']].dropna()

corr_with_score = df_corr.corr()['score'].drop('score').sort_values()

rename_map = {
    'precio':       'Precio',
    'n_servicios':  'N.º Servicios',
    'n_palabras':   'Longitud comentario',
    'has_Piscina':  'Piscina',
    'has_Spa':      'Spa/Fitness',
    'has_Desayuno': 'Desayuno',
    'has_Parking':  'Parking',
    'has_Terraza':  'Terraza/Jardín',
}
corr_with_score.index = [rename_map.get(i, i) for i in corr_with_score.index]

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('E6 — Visión Unificada: Correlación Features Estructurales → Score',
             fontsize=14, fontweight='bold')

colors_corr = ['#4CAF50' if v >= 0 else '#F44336' for v in corr_with_score.values]
bars = ax.barh(corr_with_score.index, corr_with_score.values, color=colors_corr, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coeficiente de correlación con Score')
ax.set_title(
    'Verde = a mayor valor de la feature → mayor score | Rojo = relación inversa\n'
    'La longitud del comentario es el predictor estructural no textual más fuerte (y negativo)',
    fontsize=9, color='grey')

for bar, val in zip(bars, corr_with_score.values):
    ax.text(val + (0.003 if val >= 0 else -0.003),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9,
            ha='left' if val >= 0 else 'right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'E6_correlacion_unificada.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Fig E6 guardada ✓")

---
## Bloque F — Checklist Pre-Modelo

Resumen de todas las decisiones necesarias antes del entrenamiento del clasificador de sentimiento.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CHECKLIST FINAL
# ══════════════════════════════════════════════════════════════════════════
n_neg = (df_r['sentimiento'] == 'Negativo').sum()
n_neu = (df_r['sentimiento'] == 'Neutro').sum()
n_pos = (df_r['sentimiento'] == 'Positivo').sum()

print("=" * 68)
print("  CHECKLIST PRE-MODELO — Resultados EDA Completa")
print("=" * 68)
print(f"""
1. VARIABLE OBJETIVO
   ✓ Elección: 3 clases (Negativo ≤6 / Neutro 7-8 / Positivo 9-10)
   ✓ Distribución: {n_neg} / {n_neu} / {n_pos}
              ({n_neg/len(df_r)*100:.1f}% / {n_neu/len(df_r)*100:.1f}% / {n_pos/len(df_r)*100:.1f}%)
   ✓ Ratio máximo: {n_pos//n_neg}:1 (Positivo:Negativo)
   → ACCIÓN: usar class_weight='balanced' o oversampling de negativos

2. ANOMALÍA BOOKING
   ⚠️  Booking tiene 0 reseñas negativas en el dataset.
   → ACCIÓN: excluir 'fuente' como feature directa o tratar Booking
     por separado en la evaluación del modelo.

3. FEATURES ESTRUCTURALES RECOMENDADAS
   ✓ Precio          (correlación positiva con score)  → incluir
   ✓ Longitud texto  (correlación negativa fuerte)     → incluir siempre
   ✓ N.º servicios   (correlación débil)               → incluir con cautela
   ✓ Piscina/Spa     (efecto leve pero consistente)    → incluir
   ⚠️  Parking        (efecto confusor, inverso)        → analizar antes de incluir

4. MADRID vs BARCELONA vs BILBAO
   Madrid tiene el 91.9% de reseñas positivas vs ~79-80% en las otras ciudades.
   Causa probable: mayor presencia de Booking en Madrid.
   → ACCIÓN: usar ciudad como variable de estratificación en train/test split.

5. CALIDAD DEL TEXTO
   Negativos: media {df_r[df_r['sentimiento']=='Negativo']['n_palabras'].mean():.1f} palabras
   Neutros:   media {df_r[df_r['sentimiento']=='Neutro']['n_palabras'].mean():.1f} palabras
   Positivos: media {df_r[df_r['sentimiento']=='Positivo']['n_palabras'].mean():.1f} palabras
   → Los negativos son mucho más largos: útil como feature adicional.

6. OUTLIERS DE PRECIO
   {df_p['es_outlier'].sum()} hoteles por encima del límite IQR ({upper_fence:.0f}€).
   → ACCIÓN: aplicar winsorización o excluirlos del entrenamiento.
""")

print("─" * 68)
print("  FIGURAS GENERADAS")
print("─" * 68)
figs = [
    ('B1','Visión general del corpus'),
    ('B2','Distribución de scores'),
    ('B3','Evolución temporal'),
    ('B4','Longitud de textos'),
    ('B5','N-gramas (bigramas)'),
    ('B6','Word clouds por ciudad'),
    ('B7','Modelado de tópicos LDA'),
    ('B8','TF-IDF correlación con score'),
    ('B9','Tópicos por ciudad'),
    ('C1','Visión general de servicios'),
    ('C2','Top 25 servicios por frecuencia'),
    ('C3','Prevalencia de macro-categorías'),
    ('C4','Heatmap categorías por ciudad'),
    ('C5','Servicios vs rating'),
    ('C6','Word cloud de servicios'),
    ('D1','Distribución de precios'),
    ('D2','Precio vs rating'),
    ('D3','Outliers y violín'),
    ('E1','Definición de la variable objetivo'),
    ('E2','Balance del dataset'),
    ('E3','Precio → sentimiento'),
    ('E4','Servicios → sentimiento'),
    ('E5','Calidad del texto por clase'),
    ('E6','Visión unificada: correlación features → score'),
]
for code, desc in figs:
    print(f"  {code}  {desc}")
print(f"\n✅ Todas las figuras guardadas en: {OUTPUT_DIR}")